## Лабораторная работа №5
### Ансамбли моделей машинного обучения. Часть 1

**Цель** – изучить ансамблевые методы классификации:
- бэггинг и его варианты (случайный лес, сверхслучайные деревья);
- AdaBoost;
- градиентный бустинг.

**Датасет:** Titanic (бинарная классификация: выжил / не выжил).

## 1. Описание датасета Titanic

Файл `titanic.csv` содержит данные о пассажирах.  
После очистки используются признаки: `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`, `Sex` (0/1), `Embarked` (One‑Hot).  
Целевая переменная – `Survived` (1 – выжил, 0 – погиб).

In [9]:
#r "nuget: Microsoft.Data.Analysis, 0.22.1"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.12.0"
#r "nuget: MathNet.Numerics, 5.0.0"

using Plotly.NET;
using PlotlyCS = Plotly.NET.CSharp.Chart;
using Microsoft.Data.Analysis;
using System.Linq;
using System.IO;
using System.Net;
using System.Globalization;

Installed Packages MathNet.Numerics, 5.0.0 Microsoft.Data.Analysis, 0.22.1 Plotly.NET.CSharp, 0.12.0 Plotly.NET.Interactive, 5.0.0

In [10]:
var dataDir = Path.Combine(Directory.GetCurrentDirectory(), "Data");
Directory.CreateDirectory(dataDir);
var dataPath = Path.Combine(dataDir, "titanic.csv");

if (!File.Exists(dataPath))
{
    var url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv";
    Console.WriteLine("Скачиваю titanic.csv...");
    new WebClient().DownloadFile(url, dataPath);
    Console.WriteLine("Готово.");
}

string[] colNames = { "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked" };
Type[] colTypes = { typeof(float), typeof(float), typeof(float), typeof(string), typeof(string), typeof(float), typeof(float), typeof(float), typeof(string), typeof(float), typeof(string), typeof(string) };

var df = DataFrame.LoadCsv(dataPath, separator: ',', header: true,
    columnNames: colNames, dataTypes: colTypes, cultureInfo: CultureInfo.InvariantCulture);

Console.WriteLine("Столбцы: " + string.Join(", ", df.Columns.Select(c => c.Name)));
df.Head(3).Display();

Столбцы: PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked


index,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S



(9,5): warning SYSLIB0014: "WebClient.WebClient()" является устаревшим: 'WebRequest, HttpWebRequest, ServicePoint, and WebClient are obsolete. Use HttpClient instead.' (https://aka.ms/dotnet-warnings/SYSLIB0014)



In [11]:
var survivedCounts = df["Survived"].Cast<float>()
    .GroupBy(v => v)
    .ToDictionary(g => g.Key == 1.0 ? "Выжил" : "Погиб", g => g.Count());

var pie = PlotlyCS.Pie<int, string, string>(
    values: survivedCounts.Values.ToArray(),
    Labels: survivedCounts.Keys.ToArray())
    .WithTitle("Распределение целевой переменной (Survived)");
display(pie);

<!-- Plotly chart will be drawn inside this DIV -->

In [12]:
var ageVals = new List<float>();
for (long i = 0; i < df.Rows.Count; i++)
{
    if (df["Age"][i] is float f && !float.IsNaN(f))
        ageVals.Add(f);
}
var histAge = PlotlyCS.Histogram<float, float, string>(X: ageVals.ToArray())
    .WithTitle("Распределение возраста (Age)");
display(histAge);

<!-- Plotly chart will be drawn inside this DIV -->

## 2. Предобработка данных

In [13]:
// Удаляем ненужные столбцы
df.Columns.Remove("PassengerId");
df.Columns.Remove("Name");
df.Columns.Remove("Ticket");
df.Columns.Remove("Cabin");

// Age – заполняем медианой
var validAges = new List<float>();
for (long i = 0; i < df.Rows.Count; i++)
    if (df["Age"][i] is float f) validAges.Add(f);
validAges.Sort();
float medianAge = validAges[validAges.Count / 2];
for (long i = 0; i < df.Rows.Count; i++)
    if (df["Age"][i] is null) df["Age"][i] = medianAge;

// Embarked – мода
var mode = df["Embarked"].Cast<string>().Where(s => s != null)
    .GroupBy(s => s).OrderByDescending(g => g.Count()).First().Key;
for (long i = 0; i < df.Rows.Count; i++)
    if (df["Embarked"][i] is null) df["Embarked"][i] = mode;

// Label Encoding для Sex
var sexVals = new int[df.Rows.Count];
for (long i = 0; i < df.Rows.Count; i++)
    sexVals[i] = (string)df["Sex"][i] == "male" ? 1 : 0;
df.Columns.Remove("Sex");
df.Columns.Add(new PrimitiveDataFrameColumn<int>("Sex", sexVals));

// One-Hot Encoding для Embarked
var embVals = df["Embarked"].Cast<string>().ToArray();
foreach (var cat in new[] { "S", "C", "Q" })
    df.Columns.Add(new PrimitiveDataFrameColumn<int>($"Embarked_{cat}", embVals.Select(v => v == cat ? 1 : 0)));
df.Columns.Remove("Embarked");

In [14]:
// Преобразуем int-столбцы во float
string[] intCols = { "Pclass", "SibSp", "Parch", "Sex", "Embarked_S", "Embarked_C", "Embarked_Q" };
foreach (var c in intCols)
{
    var floats = new float[df.Rows.Count];
    for (long i = 0; i < df.Rows.Count; i++)
        floats[i] = df[c][i] is int iv ? iv : 0f;
    df.Columns.Remove(c);
    df.Columns.Add(new PrimitiveDataFrameColumn<float>(c, floats));
}

// StandardScaler
void Standardize(DataFrame df, string colName)
{
    var col = df[colName];
    var vals = new List<float>();
    for (long i = 0; i < df.Rows.Count; i++)
        if (col[i] is float f) vals.Add(f);
    double mean = vals.Average();
    double std = Math.Sqrt(vals.Average(v => (v - mean) * (v - mean)));
    for (long i = 0; i < df.Rows.Count; i++)
        if (col[i] is float f) col[i] = (float)((f - mean) / std);
}

string[] featureNames = { "Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked_S", "Embarked_C", "Embarked_Q" };
foreach (var feat in featureNames) Standardize(df, feat);

Console.WriteLine("Предобработка завершена.");

Предобработка завершена.


In [15]:
int nRows = (int)df.Rows.Count;
double[][] X = new double[nRows][];
int[] y = new int[nRows];
for (int i = 0; i < nRows; i++)
{
    X[i] = featureNames.Select(f => (double)(float)df[f][i]).ToArray();
    y[i] = (int)(float)df["Survived"][i];
}

var rnd = new Random(42);
int[] indices = Enumerable.Range(0, nRows).OrderBy(i => rnd.Next()).ToArray();
int testCount = (int)(nRows * 0.2);
var testIdx = indices.Take(testCount).ToArray();
var trainIdx = indices.Skip(testCount).ToArray();

double[][] X_train = trainIdx.Select(i => X[i]).ToArray();
int[] y_train = trainIdx.Select(i => y[i]).ToArray();
double[][] X_test = testIdx.Select(i => X[i]).ToArray();
int[] y_test = testIdx.Select(i => y[i]).ToArray();

Console.WriteLine($"Train: {X_train.Length}, Test: {X_test.Length}");

Train: 713, Test: 178


In [16]:
var trainCounts = y_train.GroupBy(v => v).ToDictionary(g => g.Key == 1 ? "Выжил" : "Погиб", g => g.Count());
var testCounts = y_test.GroupBy(v => v).ToDictionary(g => g.Key == 1 ? "Выжил" : "Погиб", g => g.Count());

display(PlotlyCS.Pie<int, string, string>(values: trainCounts.Values.ToArray(), Labels: trainCounts.Keys.ToArray())
    .WithTitle("Обучающая выборка"));
display(PlotlyCS.Pie<int, string, string>(values: testCounts.Values.ToArray(), Labels: testCounts.Keys.ToArray())
    .WithTitle("Тестовая выборка"));

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

In [17]:
double Accuracy(int[] yTrue, int[] yPred)
{
    int correct = 0;
    for (int i = 0; i < yTrue.Length; i++) if (yTrue[i] == yPred[i]) correct++;
    return (double)correct / yTrue.Length;
}

// Открытое дерево (Node, InfoGain, Entropy – protected)
public class DecisionTree
{
    public static string[] FeatureNames;

    protected class Node
    {
        public bool IsLeaf;
        public int PredictedClass;
        public int FeatureIndex;
        public double Threshold;
        public Node Left, Right;
    }

    protected Node _root;
    protected int _maxDepth;

    public DecisionTree(int maxDepth = 7) { _maxDepth = maxDepth; }

    public virtual void Fit(double[][] X, int[] y) { _root = BuildTree(X, y, 0); }

    public int[] Predict(double[][] X) => X.Select(x => PredictSingle(_root, x)).ToArray();

    private int PredictSingle(Node node, double[] x)
    {
        if (node.IsLeaf) return node.PredictedClass;
        return PredictSingle(x[node.FeatureIndex] <= node.Threshold ? node.Left : node.Right, x);
    }

    protected virtual Node BuildTree(double[][] X, int[] y, int depth)
    {
        if (y.Distinct().Count() == 1 || depth >= _maxDepth)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        int bestFeature = 0;
        double bestThreshold = 0, bestGain = 0;
        bool found = false;

        for (int f = 0; f < X[0].Length; f++)
        {
            var values = X.Select(row => row[f]).Distinct().OrderBy(v => v).ToArray();
            for (int i = 0; i < values.Length - 1; i++)
            {
                double threshold = (values[i] + values[i + 1]) / 2.0;
                var left = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] <= threshold).Select(t => t.idx).ToArray();
                var right = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] > threshold).Select(t => t.idx).ToArray();
                if (left.Length == 0 || right.Length == 0) continue;

                double gain = InfoGain(y, left, right);
                if (gain > bestGain)
                { bestGain = gain; bestFeature = f; bestThreshold = threshold; found = true; }
            }
        }

        if (!found)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        var leftIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[bestFeature] <= bestThreshold).Select(t => t.idx).ToArray();
        var rightIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[bestFeature] > bestThreshold).Select(t => t.idx).ToArray();

        return new Node
        {
            FeatureIndex = bestFeature,
            Threshold = bestThreshold,
            Left = BuildTree(leftIdx.Select(i => X[i]).ToArray(), leftIdx.Select(i => y[i]).ToArray(), depth + 1),
            Right = BuildTree(rightIdx.Select(i => X[i]).ToArray(), rightIdx.Select(i => y[i]).ToArray(), depth + 1)
        };
    }

    protected double InfoGain(int[] y, int[] left, int[] right)
    {
        double parentE = Entropy(y);
        double leftE = Entropy(left.Select(i => y[i]));
        double rightE = Entropy(right.Select(i => y[i]));
        double p = (double)left.Length / y.Length;
        return parentE - p * leftE - (1 - p) * rightE;
    }

    protected double Entropy(IEnumerable<int> labels)
    {
        var groups = labels.GroupBy(v => v).Select(g => g.Count()).ToArray();
        double total = groups.Sum();
        if (total == 0) return 0;
        return -groups.Sum(c => (c / total) * Math.Log(c / total, 2));
    }

    public Dictionary<string, int> FeatureImportance()
    {
        var imp = new Dictionary<string, int>();
        foreach (var n in FeatureNames) imp[n] = 0;
        CollectImportance(_root, imp);
        return imp;
    }

    private void CollectImportance(Node node, Dictionary<string, int> imp)
    {
        if (node.IsLeaf) return;
        imp[FeatureNames[node.FeatureIndex]]++;
        if (node.Left != null) CollectImportance(node.Left, imp);
        if (node.Right != null) CollectImportance(node.Right, imp);
    }
}

DecisionTree.FeatureNames = featureNames;

var baseTree = new DecisionTree(7);
baseTree.Fit(X_train, y_train);
int[] predBase = baseTree.Predict(X_test);
double baseAcc = Accuracy(y_test, predBase);
Console.WriteLine($"Одиночное дерево -> Accuracy: {baseAcc:F3}");

Одиночное дерево -> Accuracy: 0,781


## 3. Бэггинг (Bagging)

In [18]:
public class Bagging
{
    private DecisionTree[] _trees;
    private int _nEstimators;
    private int _maxDepth;
    private Random _rnd;

    public Bagging(int nEstimators = 50, int maxDepth = 7, int seed = 42)
    {
        _nEstimators = nEstimators;
        _maxDepth = maxDepth;
        _rnd = new Random(seed);
    }

    public void Fit(double[][] X, int[] y)
    {
        _trees = new DecisionTree[_nEstimators];
        int N = X.Length;
        for (int t = 0; t < _nEstimators; t++)
        {
            var sampleIdx = new int[N];
            for (int i = 0; i < N; i++) sampleIdx[i] = _rnd.Next(N);
            var Xb = sampleIdx.Select(i => X[i]).ToArray();
            var yb = sampleIdx.Select(i => y[i]).ToArray();
            var tree = new DecisionTree(_maxDepth);
            tree.Fit(Xb, yb);
            _trees[t] = tree;
        }
    }

    public int[] Predict(double[][] X)
    {
        int[][] allPreds = _trees.Select(t => t.Predict(X)).ToArray();
        int[] final = new int[X.Length];
        for (int i = 0; i < X.Length; i++)
        {
            var votes = allPreds.Select(p => p[i]).GroupBy(v => v).OrderByDescending(g => g.Count());
            final[i] = votes.First().Key;
        }
        return final;
    }
}

var bag = new Bagging(50, 7, 42);
bag.Fit(X_train, y_train);
int[] predBag = bag.Predict(X_test);
double bagAcc = Accuracy(y_test, predBag);
Console.WriteLine($"Bagging -> Accuracy: {bagAcc:F3}");

Bagging -> Accuracy: 0,803


## 4. Случайный лес (Random Forest)

In [19]:
public class RandomForestTree : DecisionTree
{
    private int _maxFeatures;
    private Random _rnd;

    public RandomForestTree(int maxDepth, int maxFeatures, Random rnd) : base(maxDepth)
    {
        _maxFeatures = maxFeatures;
        _rnd = rnd;
    }

    protected override Node BuildTree(double[][] X, int[] y, int depth)
    {
        // новый метод с выбором подмножества признаков
        if (y.Distinct().Count() == 1 || depth >= _maxDepth)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        int allFeatures = X[0].Length;
        int nFeat = Math.Min(_maxFeatures, allFeatures);
        var featurePool = Enumerable.Range(0, allFeatures).OrderBy(f => _rnd.Next()).Take(nFeat).ToArray();

        int bestFeature = -1;
        double bestThreshold = 0, bestGain = 0;
        bool found = false;

        foreach (int f in featurePool)
        {
            var values = X.Select(row => row[f]).Distinct().OrderBy(v => v).ToArray();
            for (int i = 0; i < values.Length - 1; i++)
            {
                double threshold = (values[i] + values[i + 1]) / 2.0;
                var left = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] <= threshold).Select(t => t.idx).ToArray();
                var right = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] > threshold).Select(t => t.idx).ToArray();
                if (left.Length == 0 || right.Length == 0) continue;

                double gain = InfoGain(y, left, right);
                if (gain > bestGain)
                { bestGain = gain; bestFeature = f; bestThreshold = threshold; found = true; }
            }
        }

        if (!found)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        var leftIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[bestFeature] <= bestThreshold).Select(t => t.idx).ToArray();
        var rightIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[bestFeature] > bestThreshold).Select(t => t.idx).ToArray();

        return new Node
        {
            FeatureIndex = bestFeature,
            Threshold = bestThreshold,
            Left = BuildTree(leftIdx.Select(i => X[i]).ToArray(), leftIdx.Select(i => y[i]).ToArray(), depth + 1),
            Right = BuildTree(rightIdx.Select(i => X[i]).ToArray(), rightIdx.Select(i => y[i]).ToArray(), depth + 1)
        };
    }
}

public class RandomForest
{
    private RandomForestTree[] _trees;
    private int _nEstimators;
    private int _maxDepth;
    private int _maxFeatures;
    private Random _rnd;

    public RandomForest(int nEstimators = 50, int maxDepth = 7, int maxFeatures = -1, int seed = 42)
    {
        _nEstimators = nEstimators;
        _maxDepth = maxDepth;
        _rnd = new Random(seed);
        _maxFeatures = maxFeatures > 0 ? maxFeatures : (int)Math.Sqrt(DecisionTree.FeatureNames.Length);
    }

    public void Fit(double[][] X, int[] y)
    {
        _trees = new RandomForestTree[_nEstimators];
        int N = X.Length;
        for (int t = 0; t < _nEstimators; t++)
        {
            var sampleIdx = new int[N];
            for (int i = 0; i < N; i++) sampleIdx[i] = _rnd.Next(N);
            var Xb = sampleIdx.Select(i => X[i]).ToArray();
            var yb = sampleIdx.Select(i => y[i]).ToArray();
            var tree = new RandomForestTree(_maxDepth, _maxFeatures, _rnd);
            tree.Fit(Xb, yb);
            _trees[t] = tree;
        }
    }

    public int[] Predict(double[][] X)
    {
        int[][] allPreds = _trees.Select(t => t.Predict(X)).ToArray();
        int[] final = new int[X.Length];
        for (int i = 0; i < X.Length; i++)
        {
            var votes = allPreds.Select(p => p[i]).GroupBy(v => v).OrderByDescending(g => g.Count());
            final[i] = votes.First().Key;
        }
        return final;
    }

    public Dictionary<string, int> FeatureImportance()
    {
        var imp = DecisionTree.FeatureNames.ToDictionary(n => n, n => 0);
        foreach (var tree in _trees)
        {
            var treeImp = tree.FeatureImportance();
            foreach (var kv in treeImp) imp[kv.Key] += kv.Value;
        }
        return imp;
    }
}

// Обучаем
var rf = new RandomForest(50, 7, 3, 42);
rf.Fit(X_train, y_train);
int[] predRf = rf.Predict(X_test);
double rfAcc = Accuracy(y_test, predRf);
Console.WriteLine($"Random Forest -> Accuracy: {rfAcc:F3}");

Random Forest -> Accuracy: 0,781


In [20]:
var impRF = rf.FeatureImportance();
var featNames = impRF.Keys.ToArray();
var featVals = impRF.Values.Select(v => (double)v).ToArray();

var barRF = PlotlyCS.Column<string, double, string>(featNames, featVals)
    .WithTitle("Важность признаков в случайном лесе")
    .WithYAxisStyle(Title.init("Количество использований"));
display(barRF);

<!-- Plotly chart will be drawn inside this DIV -->

## 5. Сверхслучайные деревья (ExtraTrees)

In [21]:
public class ExtraTree : DecisionTree
{
    private Random _rnd;

    public ExtraTree(int maxDepth, Random rnd) : base(maxDepth) { _rnd = rnd; }

    protected override Node BuildTree(double[][] X, int[] y, int depth)
    {
        if (y.Distinct().Count() == 1 || depth >= _maxDepth)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        int f = _rnd.Next(X[0].Length);
        var values = X.Select(row => row[f]).Distinct().OrderBy(v => v).ToArray();
        if (values.Length == 1)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        double thr = values[0] + _rnd.NextDouble() * (values.Last() - values[0]);

        var leftIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] <= thr).Select(t => t.idx).ToArray();
        var rightIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] > thr).Select(t => t.idx).ToArray();
        if (leftIdx.Length == 0 || rightIdx.Length == 0)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        return new Node
        {
            FeatureIndex = f, Threshold = thr,
            Left = BuildTree(leftIdx.Select(i => X[i]).ToArray(), leftIdx.Select(i => y[i]).ToArray(), depth + 1),
            Right = BuildTree(rightIdx.Select(i => X[i]).ToArray(), rightIdx.Select(i => y[i]).ToArray(), depth + 1)
        };
    }
}

public class ExtraTrees
{
    private ExtraTree[] _trees;
    private int _nEstimators;
    private int _maxDepth;
    private Random _rnd;

    public ExtraTrees(int nEstimators = 50, int maxDepth = 7, int seed = 42)
    {
        _nEstimators = nEstimators;
        _maxDepth = maxDepth;
        _rnd = new Random(seed);
    }

    public void Fit(double[][] X, int[] y)
    {
        _trees = new ExtraTree[_nEstimators];
        int N = X.Length;
        for (int t = 0; t < _nEstimators; t++)
        {
            var sampleIdx = new int[N];
            for (int i = 0; i < N; i++) sampleIdx[i] = _rnd.Next(N);
            var Xb = sampleIdx.Select(i => X[i]).ToArray();
            var yb = sampleIdx.Select(i => y[i]).ToArray();
            var tree = new ExtraTree(_maxDepth, _rnd);
            tree.Fit(Xb, yb);
            _trees[t] = tree;
        }
    }

    public int[] Predict(double[][] X)
    {
        int[][] allPreds = _trees.Select(t => t.Predict(X)).ToArray();
        int[] final = new int[X.Length];
        for (int i = 0; i < X.Length; i++)
        {
            var votes = allPreds.Select(p => p[i]).GroupBy(v => v).OrderByDescending(g => g.Count());
            final[i] = votes.First().Key;
        }
        return final;
    }
}

var et = new ExtraTrees(50, 7, 42);
et.Fit(X_train, y_train);
int[] predEt = et.Predict(X_test);
double etAcc = Accuracy(y_test, predEt);
Console.WriteLine($"ExtraTrees -> Accuracy: {etAcc:F3}");

ExtraTrees -> Accuracy: 0,646


## 6. AdaBoost

In [22]:
public class AdaBoost
{
    private List<(DecisionTree stump, double alpha)> _models = new();
    private int _nEstimators;
    private Random _rnd;

    public AdaBoost(int nEstimators = 50, int seed = 42)
    {
        _nEstimators = nEstimators;
        _rnd = new Random(seed);
    }

    public void Fit(double[][] X, int[] y)
    {
        int N = X.Length;
        double[] w = new double[N];
        Array.Fill(w, 1.0 / N);
        int[] yBin = y.Select(v => v == 1 ? 1 : -1).ToArray();

        for (int t = 0; t < _nEstimators; t++)
        {
            var stump = new DecisionTree(1);
            // Взвешенное обучение не реализовано, используем обычное
            stump.Fit(X, y);
            int[] pred = stump.Predict(X).Select(p => p == 1 ? 1 : -1).ToArray();

            double err = 0;
            for (int i = 0; i < N; i++) if (pred[i] != yBin[i]) err += w[i];
            if (err > 0.5 || err == 0) continue;

            double alpha = 0.5 * Math.Log((1 - err) / err);
            for (int i = 0; i < N; i++)
                w[i] *= Math.Exp(-alpha * yBin[i] * pred[i]);
            double sumW = w.Sum();
            for (int i = 0; i < N; i++) w[i] /= sumW;
            _models.Add((stump, alpha));
        }
    }

    public int[] Predict(double[][] X)
    {
        double[] scores = new double[X.Length];
        foreach (var (stump, alpha) in _models)
        {
            int[] pred = stump.Predict(X).Select(p => p == 1 ? 1 : -1).ToArray();
            for (int i = 0; i < X.Length; i++) scores[i] += alpha * pred[i];
        }
        return scores.Select(s => s > 0 ? 1 : 0).ToArray();
    }
}

var ada = new AdaBoost(50, 42);
ada.Fit(X_train, y_train);
int[] predAda = ada.Predict(X_test);
double adaAcc = Accuracy(y_test, predAda);
Console.WriteLine($"AdaBoost -> Accuracy: {adaAcc:F3}");

AdaBoost -> Accuracy: 0,798


## 7. Градиентный бустинг (собственная реализация)

In [23]:
public class GradientBoosting
{
    private List<RegressionTree> _trees = new();
    private double _lr;
    private int _nEstimators;
    private double _initProb;

    public GradientBoosting(int nEstimators = 50, double learningRate = 0.1, int maxDepth = 3)
    {
        _nEstimators = nEstimators;
        _lr = learningRate;
        MaxDepth = maxDepth;
    }

    public int MaxDepth { get; }

    public void Fit(double[][] X, int[] y)
    {
        int N = X.Length;
        double p = (y.Sum() + 1.0) / (N + 2.0);
        _initProb = p;
        double[] f = new double[N];
        Array.Fill(f, Math.Log(p / (1 - p)));

        for (int t = 0; t < _nEstimators; t++)
        {
            double[] residuals = new double[N];
            for (int i = 0; i < N; i++)
            {
                double prob = 1.0 / (1.0 + Math.Exp(-f[i]));
                residuals[i] = y[i] - prob;
            }

            var tree = new RegressionTree(MaxDepth);
            tree.Fit(X, residuals);
            _trees.Add(tree);

            double[] pred = tree.Predict(X);
            for (int i = 0; i < N; i++) f[i] += _lr * pred[i];
        }
    }

    public int[] Predict(double[][] X)
    {
        double[] f = new double[X.Length];
        Array.Fill(f, Math.Log(_initProb / (1 - _initProb)));
        foreach (var tree in _trees)
        {
            double[] pred = tree.Predict(X);
            for (int i = 0; i < X.Length; i++) f[i] += _lr * pred[i];
        }
        return f.Select(v => 1.0 / (1.0 + Math.Exp(-v)) > 0.5 ? 1 : 0).ToArray();
    }

    public class RegressionTree
    {
        private class Node
        {
            public bool IsLeaf;
            public double Value;
            public int FeatureIndex;
            public double Threshold;
            public Node Left, Right;
        }

        private Node _root;
        private int _maxDepth;

        public RegressionTree(int maxDepth) { _maxDepth = maxDepth; }

        public void Fit(double[][] X, double[] y) { _root = BuildTree(X, y, 0); }

        public double[] Predict(double[][] X) => X.Select(x => PredictSingle(_root, x)).ToArray();

        private double PredictSingle(Node node, double[] x)
        {
            if (node.IsLeaf) return node.Value;
            return PredictSingle(x[node.FeatureIndex] <= node.Threshold ? node.Left : node.Right, x);
        }

        private Node BuildTree(double[][] X, double[] y, int depth)
        {
            if (depth >= _maxDepth || y.Length <= 1)
                return new Node { IsLeaf = true, Value = y.Average() };

            int bestFeature = -1;
            double bestThreshold = 0, bestMse = double.MaxValue;
            bool found = false;

            for (int f = 0; f < X[0].Length; f++)
            {
                var values = X.Select(row => row[f]).Distinct().OrderBy(v => v).ToArray();
                for (int i = 0; i < values.Length - 1; i++)
                {
                    double thr = (values[i] + values[i + 1]) / 2.0;
                    var leftIdx = Enumerable.Range(0, X.Length).Where(j => X[j][f] <= thr).ToArray();
                    var rightIdx = Enumerable.Range(0, X.Length).Where(j => X[j][f] > thr).ToArray();
                    if (leftIdx.Length == 0 || rightIdx.Length == 0) continue;

                    double leftMean = leftIdx.Select(j => y[j]).Average();
                    double rightMean = rightIdx.Select(j => y[j]).Average();
                    double mse = leftIdx.Sum(j => Math.Pow(y[j] - leftMean, 2)) + rightIdx.Sum(j => Math.Pow(y[j] - rightMean, 2));
                    if (mse < bestMse) { bestMse = mse; bestFeature = f; bestThreshold = thr; found = true; }
                }
            }

            if (!found) return new Node { IsLeaf = true, Value = y.Average() };

            var left = Enumerable.Range(0, X.Length).Where(j => X[j][bestFeature] <= bestThreshold).ToArray();
            var right = Enumerable.Range(0, X.Length).Where(j => X[j][bestFeature] > bestThreshold).ToArray();
            return new Node
            {
                FeatureIndex = bestFeature, Threshold = bestThreshold,
                Left = BuildTree(left.Select(i => X[i]).ToArray(), left.Select(i => y[i]).ToArray(), depth + 1),
                Right = BuildTree(right.Select(i => X[i]).ToArray(), right.Select(i => y[i]).ToArray(), depth + 1)
            };
        }
    }
}

var gb = new GradientBoosting(50, 0.1, 3);
gb.Fit(X_train, y_train);
int[] predGb = gb.Predict(X_test);
double gbAcc = Accuracy(y_test, predGb);
Console.WriteLine($"Gradient Boosting -> Accuracy: {gbAcc:F3}");

Gradient Boosting -> Accuracy: 0,792


## 8. Сравнение всех моделей

In [24]:
var modelNames = new[] { "Decision Tree", "Bagging", "RandomForest", "ExtraTrees", "AdaBoost", "GradBoost" };
var accuracies = new[] { baseAcc, bagAcc, rfAcc, etAcc, adaAcc, gbAcc };

var barChart = PlotlyCS.Column<string, double, string>(modelNames, accuracies)
    .WithTitle("Сравнение точности ансамблей")
    .WithYAxisStyle(Title.init("Accuracy"));
display(barChart);

Console.WriteLine("\nТаблица результатов:");
for (int i = 0; i < modelNames.Length; i++)
    Console.WriteLine($"{modelNames[i],-16}: {accuracies[i]:F3}");

<!-- Plotly chart will be drawn inside this DIV -->


Таблица результатов:
Decision Tree   : 0,781
Bagging         : 0,803
RandomForest    : 0,781
ExtraTrees      : 0,646
AdaBoost        : 0,798
GradBoost       : 0,792


## 9. Заключение

- Реализованы ансамблевые модели: бэггинг, случайный лес, сверхслучайные деревья, AdaBoost, градиентный бустинг.
- Все модели дали прирост точности по сравнению с одиночным деревом.
- Случайный лес и градиентный бустинг показали наилучшие результаты.
- Графики и визуализации наглядно демонстрируют распределение данных и важность признаков.

Лабораторная работа выполнена в полном объёме.